# TLIO: Tight Learned Inertial Odometry
## Comprehensive Mathematical Documentation

**GPS-Free Drone Navigation using Neural Networks + Extended Kalman Filter**

This notebook provides complete mathematical documentation for the TLIO system, from raw IMU data to final position estimates.

---

## Table of Contents

1. **System Overview** - High-level architecture and GHOST value proposition
2. **Mathematical Foundation** - Neural network architecture, loss functions, uncertainty quantification
3. **Coordinate Transformations** - Quaternions, body/world frames, rotation matrices
4. **Extended Kalman Filter** - State vector, prediction, measurement updates
5. **Data Pipeline** - Sliding windows, real IMU sequences, label creation
6. **Training & Performance** - Loss curves, metrics, drift characteristics
7. **Architecture Comparison** - v1 vs v2 vs v2.5 performance and trade-offs
8. **Hardware Deployment** - ArduPilot integration, configuration, real-world impact

# 1. System Overview

## The Big Idea
```
Your drone starts at position (0, 0, 0). Every second, the TLIO model says "you moved (dx, dy, dz)". 
The EKF accumulates these into a running position. That's it!

Time 0s:  Position = (0, 0, 0)                        ← starting point
Time 1s:  Position = (0, 0, 0) + (0.5, 0.1, 0.2)      = (0.5, 0.1, 0.2)
Time 2s:  Position = (0.5, 0.1, 0.2) + (0.3, -0.1, 0.1) = (0.8, 0.0, 0.3)
Time 3s:  Position = (0.8, 0.0, 0.3) + (0.4, 0.2, -0.1) = (1.2, 0.2, 0.2)
```

## Research Architecture

```
┌─────────────────────────────────────────────────────────┐
│                    YOUR DRONE                           │
│                                                         │
│  IMU reads 200 samples/sec (accel + gyro)               │
│         │                                               │
│         ▼                                               │
│  ┌─────────────────┐                                    │
│  │  LAYER 1: TLIO  │  Neural Network                    │
│  │  (model_arch)   │  Every 0.5 sec (250 samples):      │
│  │                 │  "You moved 0.5m right,            │
│  │  Input:  250×6  │   0.1m forward, 0.2m up"           │
│  │  Output: dx,dy,dz + confidence                       │
│  └────────┬────────┘                                    │
│           │  displacement every 0.5 sec (2Hz)           │
│           ▼                                             │
│  ┌─────────────────┐                                    │
│  │  LAYER 2: EKF   │  State Estimator                   │
│  │                 │                                    │
│  │  Keeps track of:│                                    │
│  │  • Position     │ ← starts at (0,0,0)                │
│  │  • Velocity     │ ← starts at (0,0,0)                │
│  │  • Orientation  │ ← starts at level                  │
│  │  • IMU biases   │ ← learns over time                 │
│  │                 │                                    │
│  │  PREDICT: 200Hz │ ← raw IMU (fast, drifts)           │
│  │  UPDATE:   2Hz  │ ← TLIO displacement (corrects!)    │
│  │  UPDATE:  10Hz  │ ← barometer (Z only)               │
│  └────────┬────────┘                                    │
│           │  position, velocity, orientation             │
│           ▼                                             │
│  ┌─────────────────┐                                    │
│  │  LAYER 3: PID   │  Controller                        │
│  │  or RL Agent    │                                    │
│  │                 │  "I'm at (1.2, 0.2, 0.2)           │
│  │                 │   I want to be at (5, 0, 1)         │
│  │                 │   → spin motors: 70%, 80%, 65%, 75%"│
│  └────────┬────────┘                                    │
│           │  motor commands                             │
│           ▼                                             │
│       ⟨ MOTORS ⟩                                        │
└─────────────────────────────────────────────────────────┘
```

## GHOST = GPS Replacement (Not EKF Replacement)

**GHOST** is the "Artificial GPS" sensor that sends position estimates to ArduPilot's EKF3, exactly like GPS does.

```
┌────────────────────────────────────────────────────────┐
│              ArduPilot EKF3 (Unchanged)                │
│                                                        │
│  Sensor Inputs:                                        │
│  ┌─────────┐     ┌─────────┐     ┌─────────┐           │
│  │   GPS   │     │  GHOST  │     │  Baro   │           │
│  │         │ OR  │         │     │         │           │
│  │ sends:  │     │ sends:  │     │ sends:  │           │
│  │ position│     │ position│     │ altitude│           │
│  └────┬────┘     └────┬────┘     └────┬────┘           │
│       │               │               │                │
│       └───────────────┴───────────────┘                │
│                       │                                │
│                       ▼                                │
│              ┌─────────────────┐                       │
│              │  ArduPilot EKF3 │                       │
│              │  (Sensor Fusion)│                       │
│              │                 │                       │
│              │  Fuses all      │                       │
│              │  sensors to     │                       │
│              │  estimate state │                       │
│              └────────┬────────┘                       │
│                       │                                │
│                       ▼                                │
│              Position, Velocity, Attitude              │
└────────────────────────────────────────────────────────┘
```

### Configuration Difference

**With GPS:**
```
EK3_SRC1_POSXY = 1   # GPS position source
EK3_SRC1_POSZ  = 1   # GPS altitude source
GPS_TYPE       = 1   # Auto-detect GPS
```

**With GHOST:**
```
# ── GHOST GPS-Free Configuration ──
AHRS_EKF_TYPE  = 3    # Use EKF3
EK3_SRC1_POSXY = 6    # ExternalNav (GHOST) — horizontal position
EK3_SRC1_POSZ  = 3    # Barometer — altitude (drift-free)
EK3_SRC1_VELXY = 0    # None (derived from position)
EK3_SRC1_VELZ  = 0    # None (derived from altitude)
EK3_SRC1_YAW   = 1    # Compass
GPS_TYPE       = 0    # No GPS
VISO_TYPE      = 1    # MAVLink vision position
EK3_GPS_TYPE   = 3    # No GPS fusion
```

**Same architecture, different position source!**

# 2. Mathematical Foundation

This section provides the complete mathematical formulation of the TLIO neural network, from input normalization to uncertainty-aware loss functions.

## 2.1 Input Data Format

The neural network receives **raw IMU measurements** at 200Hz:

- **Accelerometer**: $\mathbf{a}_{raw} = [a_x, a_y, a_z]$ in m/s²
- **Gyroscope**: $\boldsymbol{\omega}_{raw} = [\omega_x, \omega_y, \omega_z]$ in rad/s

### Normalization

To improve neural network training stability, inputs are normalized:

$$
\mathbf{a}_{norm} = \frac{\mathbf{a}_{raw}}{9.81 \text{ m/s}^2}
$$

$$
\boldsymbol{\omega}_{norm} = \frac{\boldsymbol{\omega}_{raw}}{\pi \text{ rad/s}}
$$

**Rationale:**
- Accelerometer: Dividing by gravity (9.81 m/s²) makes hovering ≈ 1.0 (instead of ≈9.81)
- Gyroscope: Dividing by π rad/s puts typical rotation rates in [-1, 1] range

### Input Window

Each training sample consists of **250 consecutive IMU readings** (1.25 seconds @ 200Hz):

$$
\mathbf{X} \in \mathbb{R}^{250 \times 6}
$$

Where each row is: $[\mathbf{a}_{norm}, \boldsymbol{\omega}_{norm}] = [a_x, a_y, a_z, \omega_x, \omega_y, \omega_z]$

### Understanding Acceleration Physics

**What the accelerometer actually measures:**

The accelerometer measures **specific force** (force per unit mass), NOT gravitational acceleration:

$$
\mathbf{a}_{measured} = \mathbf{a}_{kinematic} - \mathbf{g}
$$

Where:
- $\mathbf{a}_{kinematic}$ = actual acceleration of the drone
- $\mathbf{g}$ = gravitational acceleration

### Scenario 1: Hovering (Stationary)

**Kinematic state**: $\mathbf{v} = 0$, $\mathbf{a}_{kinematic} = 0$

**What accelerometer reads**:
$$
\mathbf{a}_{measured} = 0 - (-9.81\hat{z}) = [0, 0, 9.81]^T \text{ m/s}^2
$$

The drone is **pushing against gravity** with propeller thrust. The accelerometer feels this upward force!

### Scenario 2: Constant Velocity Flight

**Kinematic state**: $\mathbf{v} = [5, 0, 0]^T$ m/s (flying north at 5 m/s), $\mathbf{a}_{kinematic} = 0$

**What accelerometer reads**:
$$
\mathbf{a}_{measured} = 0 - (-9.81\hat{z}) = [0, 0, 9.81]^T \text{ m/s}^2
$$

**Key insight**: Same as hovering! Constant velocity means **zero acceleration**.

$$
\boxed{\text{If all accelerations} = 0 \text{ (except gravity) } \Rightarrow \text{ constant velocity!}}
$$

### Scenario 3: Accelerating Forward

**Kinematic state**: Starting from hover, accelerating forward at 2 m/s²

**What accelerometer reads** (when pitched 20° forward):
$$
\mathbf{a}_{measured} = [2.0, 0, 0] - [0, 0, -9.81] = [2.0, 0, 9.81]^T \text{ m/s}^2 \quad \text{(world frame)}
$$

But accelerometer is in **body frame** (tilted 20° forward):
$$
\mathbf{a}_{body} \approx [2.0, 0, 10.7]^T \text{ m/s}^2
$$

The forward component increases, and Z-component increases due to tilt!

### Scenario 4: Free Fall (Hypothetical)

If motors shut off and drone is falling:

**Kinematic state**: $\mathbf{a}_{kinematic} = -9.81\hat{z}$ (falling)

**What accelerometer reads**:
$$
\mathbf{a}_{measured} = -9.81\hat{z} - (-9.81\hat{z}) = [0, 0, 0]^T
$$

**Weightlessness!** Just like astronauts in orbit.

### Why This Matters for TLIO

The neural network learns patterns like:

- **Hovering**: $\mathbf{a} \approx [0, 0, 9.81]$, $\boldsymbol{\omega} \approx 0$ → $\Delta p \approx 0$
- **Constant velocity**: $\mathbf{a} \approx [0, 0, 9.81]$, $\boldsymbol{\omega} \approx 0$ → $\Delta p \approx \mathbf{v} \cdot \Delta t$
- **Accelerating**: $\mathbf{a} \neq [0, 0, 9.81]$ → Displacement includes acceleration term

**The challenge**: Distinguishing constant velocity from stationary based on IMU alone is hard! This is why:
1. Displacement accumulates drift over time
2. TLIO needs periodic corrections from the EKF
3. Barometer helps constrain Z-axis drift

## 2.2 Neural Network Architecture (v2.5)

The TLIO network uses a **Multi-Scale ResNet with Dilated Convolutions** architecture.

### Tensor Flow Through Network

```
Input:  (N, 250, 6)          # N samples, 250 timesteps, 6 IMU channels
   ↓
Conv1D (kernel=3, 64 filters)
   ↓
(N, 250, 64)                 # Initial feature extraction
   ↓
┌───────────────────────────┐
│ Residual Block 1          │
│  ┌─── Multi-Scale Branch ─┤  Parallel convolutions:
│  │    • kernel=3  → 16ch  │  - Small receptive field (high-freq)
│  │    • kernel=7  → 16ch  │  - Medium receptive field (mid-freq)
│  │    • kernel=15 → 16ch  │  - Large receptive field (low-freq)
│  │  Concat → 48ch         │
│  │                        │
│  ├─── Dilated Branch ─────┤  Dilated convolutions:
│  │    • dilation=1 → 8ch  │  - receptive field: 3 samples
│  │    • dilation=2 → 8ch  │  - receptive field: 5 samples
│  │    • dilation=4 → 8ch  │  - receptive field: 9 samples
│  │    • dilation=8 → 8ch  │  - receptive field: 17 samples
│  │  Concat → 32ch         │
│  │                        │
│  ├─── Merge ──────────────┤  Concat(multi_scale + dilated) = 80ch
│  │                        │
│  ├─── SE Attention ───────┤  Squeeze-Excitation:
│  │  GlobalAvgPool → 80    │  - Learn channel importance
│  │  Dense(20) → ReLU      │  - Suppress irrelevant features
│  │  Dense(80) → Sigmoid   │  - Amplify important features
│  │  Multiply              │
│  │                        │
│  ├─── Residual Add ───────┤  out = F(x) + x
│  └────────────────────────┘
   ↓
(N, 250, 64)                 # After residual block
   ↓
... Repeat for Block 2 ...
   ↓
GlobalAveragePooling1D       # (N, 250, 64) → (N, 64)
   ↓
Dense(32, ReLU)              # Feature compression
   ↓
Dense(6, linear)             # Final prediction
   ↓
Output: (N, 6)               # [dx, dy, dz, log_var_x, log_var_y, log_var_z]
```

### Mathematical Operations

**1D Convolution:**
$$
y[t] = \sum_{i=0}^{k-1} w[i] \cdot x[t+i] + b
$$

Where:
- $k$ = kernel size (3, 7, or 15)
- $w$ = learned weights
- $b$ = bias term

**Dilated Convolution:**
$$
y[t] = \sum_{i=0}^{k-1} w[i] \cdot x[t + i \cdot d] + b
$$

Where $d$ is the dilation rate (1, 2, 4, or 8).

**Receptive Field:**
$$
RF = 1 + \sum_{i=1}^{L} (k_i - 1) \cdot d_i
$$

For dilation rates [1, 2, 4, 8] with kernel size 3:
$$
RF = 1 + (3-1) \cdot 1 + (3-1) \cdot 2 + (3-1) \cdot 4 + (3-1) \cdot 8 = 1 + 2 + 4 + 8 + 16 = 31 \text{ samples}
$$

**Residual Connection:**
$$
\mathbf{h}_{out} = \sigma(F(\mathbf{h}_{in}; W_1, W_2)) + \mathbf{h}_{in}
$$

Where:
- $F(\cdot)$ = residual function (multi-scale + dilated + SE attention)
- $\sigma$ = activation function (ReLU)
- $+$ = element-wise addition (skip connection)

## 2.3 Output Format and Uncertainty Quantification

The network outputs **6 values** per prediction:

$$
\mathbf{y}_{pred} = [\Delta x, \Delta y, \Delta z, \log \sigma_x^2, \log \sigma_y^2, \log \sigma_z^2]
$$

### Displacement (First 3 outputs)

$$
\boldsymbol{\Delta p}_{body} = [\Delta x, \Delta y, \Delta z] \quad \text{meters in body frame}
$$

This represents **relative displacement** over the 1.25-second window, in the drone's local coordinate system:
- $\Delta x$ = forward/backward
- $\Delta y$ = left/right  
- $\Delta z$ = up/down

### Log-Variance (Last 3 outputs)

$$
\mathbf{v} = [\log \sigma_x^2, \log \sigma_y^2, \log \sigma_z^2]
$$

**Why log-variance instead of direct variance?**

1. **Numerical stability**: Variance must be positive ($\sigma^2 > 0$). Using $\log(\sigma^2)$ allows the network to output any real number, then:
   $$
   \sigma^2 = \exp(\log \sigma^2)
   $$
   
2. **Automatic positivity**: $\exp(\cdot)$ always returns positive values

3. **Better gradient flow**: Log-space helps with numerical precision during backpropagation

### Converting to Standard Deviation

$$
\sigma = \sqrt{\exp(\log \sigma^2)} = \exp\left(\frac{\log \sigma^2}{2}\right)
$$

**Interpretation:**
- Small $\sigma$ (< 0.1m) = **high confidence** → EKF trusts this prediction more
- Large $\sigma$ (> 0.5m) = **low confidence** → EKF relies more on IMU dead-reckoning

## 2.4 Loss Function: Negative Log-Likelihood (NLL)

### Why NLL Loss is Superior to Huber Loss

**Huber Loss** (robust to outliers):
$$
L_{Huber} = \begin{cases}
\frac{1}{2}(y - \hat{y})^2 & \text{if } |y - \hat{y}| \leq \delta \\
\delta|y - \hat{y}| - \frac{1}{2}\delta^2 & \text{otherwise}
\end{cases}
$$

**Problem:** Huber loss doesn't capture **uncertainty**. The network has no incentive to learn when it's confident vs uncertain.

---

**NLL Loss** (uncertainty-aware):
$$
L_{NLL} = \frac{1}{2} \left( \frac{(y - \hat{y})^2}{\sigma^2} + \log(\sigma^2) \right)
$$

Where:
- $y$ = ground truth displacement
- $\hat{y}$ = predicted displacement
- $\sigma^2 = \exp(\log \sigma^2)$ = predicted variance

### Mathematical Derivation

NLL loss comes from maximizing the likelihood of a Gaussian distribution:

$$
p(y | \hat{y}, \sigma) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y-\hat{y})^2}{2\sigma^2}\right)
$$

Taking negative log-likelihood:
$$
-\log p(y | \hat{y}, \sigma) = \frac{1}{2}\log(2\pi) + \frac{1}{2}\log(\sigma^2) + \frac{(y-\hat{y})^2}{2\sigma^2}
$$

Dropping the constant $\frac{1}{2}\log(2\pi)$:
$$
L = \frac{1}{2}\log(\sigma^2) + \frac{(y-\hat{y})^2}{2\sigma^2}
$$

### Why It Works: The Balancing Act

$$
L = \underbrace{\frac{(y-\hat{y})^2}{2\sigma^2}}_{\text{accuracy term}} + \underbrace{\frac{1}{2}\log(\sigma^2)}_{\text{penalty term}}
$$

**Two competing forces:**

1. **Accuracy term** $\frac{(y-\hat{y})^2}{2\sigma^2}$:
   - Encourages small prediction error
   - **But** also encourages large $\sigma^2$ (to reduce this term)
   
2. **Penalty term** $\frac{1}{2}\log(\sigma^2)$:
   - Penalizes large $\sigma^2$
   - **Prevents** the network from \"cheating\" with infinitely large uncertainty

**Result:** The network learns to output:
- **Small $\sigma^2$** when it's confident (and makes accurate predictions)
- **Large $\sigma^2$** when it's uncertain (difficult scenarios: rapid maneuvers, vibrations)

### Implementation (Per-Axis)

For 3D displacement, we sum losses across all axes:

$$
L_{total} = \frac{1}{2} \sum_{i \in \{x,y,z\}} \left( \frac{(\Delta p_i - \Delta \hat{p}_i)^2}{\sigma_i^2} + \log(\sigma_i^2) \right)
$$

This allows the network to learn **different uncertainties per axis** (e.g., Z-axis might be more uncertain due to altitude variations).

# 3. Coordinate Transformations

TLIO predicts displacement in **body frame** (natural for CNNs to learn from IMU data), but the EKF tracks position in **world frame** (NED: North-East-Down). Understanding these transformations is critical.

## 3.1 Quaternion Mathematics

### Definition

A **quaternion** represents 3D rotation as a 4-tuple:

$$
\mathbf{q} = [q_w, q_x, q_y, q_z] \quad \text{with} \quad \|\mathbf{q}\| = 1
$$

Where:
- $q_w$ = scalar part (real component)
- $[q_x, q_y, q_z]$ = vector part (imaginary components)

**Normalization constraint:**
$$
q_w^2 + q_x^2 + q_y^2 + q_z^2 = 1
$$

### Quaternion to Rotation Matrix

To transform vectors between frames, we convert the quaternion to a 3×3 rotation matrix:

$$
R(\mathbf{q}) = \begin{bmatrix}
1-2(q_y^2+q_z^2) & 2(q_xq_y-q_wq_z) & 2(q_xq_z+q_wq_y) \\\\
2(q_xq_y+q_wq_z) & 1-2(q_x^2+q_z^2) & 2(q_yq_z-q_wq_x) \\\\
2(q_xq_z-q_wq_y) & 2(q_yq_z+q_wq_x) & 1-2(q_x^2+q_y^2)
\end{bmatrix}
$$

**Properties:**
- $R$ is orthogonal: $R^T R = I$
- $\det(R) = 1$ (proper rotation)
- $R^{-1} = R^T$ (inverse rotation)

### Quaternion Conjugate (Inverse Rotation)

$$
\mathbf{q}^* = [q_w, -q_x, -q_y, -q_z]
$$

**Property:** $R(\mathbf{q}^*) = R(\mathbf{q})^T$

This means we can invert a rotation by simply negating the vector part of the quaternion!

## 3.2 Frame Definitions

### NED (North-East-Down) World Frame

Fixed coordinate system aligned with the Earth:
- **X-axis**: Points North
- **Y-axis**: Points East
- **Z-axis**: Points Down (into the ground)

$$
\mathbf{g}_{NED} = [0, 0, 9.81]^T \quad \text{m/s}^2
$$

Gravity points downward in NED frame.

### Body Frame (Drone-Centered)

Moving coordinate system attached to the drone:
- **X-axis**: Points forward (nose direction)
- **Y-axis**: Points right (starboard wing)
- **Z-axis**: Points down (belly)

When hovering level:
$$
\mathbf{g}_{body} = [0, 0, 9.81]^T \quad \text{m/s}^2
$$

When pitched 30° forward:
$$
\mathbf{g}_{body} \approx [4.9, 0, 8.5]^T \quad \text{m/s}^2
$$

The accelerometer measures gravity in the body frame!

### Transformation Equations

**Body → World:**
$$
\mathbf{v}_{world} = R(\mathbf{q}) \cdot \mathbf{v}_{body}
$$

**World → Body:**
$$
\mathbf{v}_{body} = R(\mathbf{q})^T \cdot \mathbf{v}_{world} = R(\mathbf{q}^*) \cdot \mathbf{v}_{world}
$$

## 3.3 Real Example: Displacement Label Creation

This is how we create training labels from ground truth data.

### Given Data

From motion capture system or SLAM, we have:

- **Start position (world frame)**: $\mathbf{p}_{start} = [0.0, 0.0, 0.0]$ m
- **End position (world frame)**: $\mathbf{p}_{end} = [0.6, 0.1, -0.05]$ m
- **Quaternion at start**: $\mathbf{q} = [0.98, 0.01, -0.15, 0.05]$ (pitched ~17° forward, slightly rolled)

### Step 1: World Frame Displacement

$$
\boldsymbol{\Delta p}_{world} = \mathbf{p}_{end} - \mathbf{p}_{start} = [0.6, 0.1, -0.05]^T \text{ m}
$$

### Step 2: Compute Rotation Matrix

Using the quaternion-to-matrix formula with $\mathbf{q} = [0.98, 0.01, -0.15, 0.05]$:

$$
R = \begin{bmatrix}
1-2(q_y^2+q_z^2) & 2(q_xq_y-q_wq_z) & 2(q_xq_z+q_wq_y) \\\\
2(q_xq_y+q_wq_z) & 1-2(q_x^2+q_z^2) & 2(q_yq_z-q_wq_x) \\\\
2(q_xq_z-q_wq_y) & 2(q_yq_z+q_wq_x) & 1-2(q_x^2+q_y^2)
\end{bmatrix}
$$

**Substituting values:**

$$
R \approx \begin{bmatrix}
0.956 & 0.098 & 0.275 \\\\
-0.099 & 0.995 & -0.010 \\\\
-0.274 & -0.039 & 0.961
\end{bmatrix}
$$

### Step 3: Transform to Body Frame

$$
\boldsymbol{\Delta p}_{body} = R^T \cdot \boldsymbol{\Delta p}_{world}
$$

$$
\boldsymbol{\Delta p}_{body} = \begin{bmatrix}
0.956 & -0.099 & -0.274 \\\\
0.098 & 0.995 & -0.039 \\\\
0.275 & -0.010 & 0.961
\end{bmatrix} \begin{bmatrix}
0.6 \\\\
0.1 \\\\
-0.05
\end{bmatrix}
$$

$$
\boldsymbol{\Delta p}_{body} = \begin{bmatrix}
0.956 \times 0.6 - 0.099 \times 0.1 - 0.274 \times (-0.05) \\\\
0.098 \times 0.6 + 0.995 \times 0.1 - 0.039 \times (-0.05) \\\\
0.275 \times 0.6 - 0.010 \times 0.1 + 0.961 \times (-0.05)
\end{bmatrix}
$$

$$
\boldsymbol{\Delta p}_{body} \approx \begin{bmatrix}
0.588 \\\\
0.161 \\\\
0.117
\end{bmatrix} \text{ meters}
$$

### Interpretation

**In the drone's perspective:**
- Moved **0.588m forward** (X-axis)
- Moved **0.161m right** (Y-axis)
- Moved **0.117m down** (Z-axis)

This $\boldsymbol{\Delta p}_{body}$ becomes the **training label** for the neural network!

**Why body frame?** The IMU measures accelerations in body frame, so it's natural for the CNN to learn patterns that correlate with body-frame displacement.

# 4. Extended Kalman Filter (EKF) Mathematics

The EKF fuses high-rate IMU measurements (200Hz) with low-rate TLIO corrections (2Hz) and barometer readings (10Hz) to maintain an optimal state estimate.

## 4.1 State Vector (16 Dimensions)

$$
\mathbf{x} = \begin{bmatrix}
\mathbf{p} \\\\
\mathbf{v} \\\\
\mathbf{q} \\\\
\mathbf{b}_a \\\\
\mathbf{b}_g
\end{bmatrix} = \begin{bmatrix}
p_x, p_y, p_z \\\\
v_x, v_y, v_z \\\\
q_w, q_x, q_y, q_z \\\\
b_{ax}, b_{ay}, b_{az} \\\\
b_{gx}, b_{gy}, b_{gz}
\end{bmatrix}
$$

**Components:**

1. **Position** $\mathbf{p} \in \mathbb{R}^3$ — NED world frame [meters]
   - $p_x$ = North position
   - $p_y$ = East position
   - $p_z$ = Down position (negative = altitude)

2. **Velocity** $\mathbf{v} \in \mathbb{R}^3$ — NED world frame [m/s]
   - $v_x, v_y, v_z$ = velocity components

3. **Orientation** $\mathbf{q} \in \mathbb{R}^4$ — Unit quaternion
   - Represents body-to-world rotation
   - Constraint: $\|\mathbf{q}\| = 1$

4. **Accelerometer Bias** $\mathbf{b}_a \in \mathbb{R}^3$ — [m/s²]
   - Slowly drifting offset in accelerometer readings
   - Estimated and compensated by EKF

5. **Gyroscope Bias** $\mathbf{b}_g \in \mathbb{R}^3$ — [rad/s]
   - Slowly drifting offset in gyroscope readings
   - Causes orientation drift if not compensated

### Initial Conditions

$$
\mathbf{x}_0 = [0, 0, 0, \quad 0, 0, 0, \quad 1, 0, 0, 0, \quad 0, 0, 0, \quad 0, 0, 0]^T
$$

- Position: origin
- Velocity: stationary
- Orientation: level ($q_w = 1$, rest zeros)
- Biases: zero (learned during flight)

## 4.2 Prediction Step (200Hz - Every IMU Reading)

The prediction step propagates the state forward using **kinematic equations** and raw IMU measurements.

### Input

- Current state: $\mathbf{x}_k$
- IMU measurement: $\mathbf{a}_{meas}, \boldsymbol{\omega}_{meas}$ (body frame)
- Time step: $\Delta t = 0.005$ seconds (200Hz)

### Bias Correction

$$
\mathbf{a} = \mathbf{a}_{meas} - \mathbf{b}_a
$$

$$
\boldsymbol{\omega} = \boldsymbol{\omega}_{meas} - \mathbf{b}_g
$$

### Orientation Update (Quaternion Integration)

$$
\mathbf{q}_{k+1} = \mathbf{q}_k + \frac{1}{2} \Omega(\boldsymbol{\omega}) \mathbf{q}_k \Delta t
$$

Where $\Omega(\boldsymbol{\omega})$ is the quaternion update matrix:

$$
\Omega(\boldsymbol{\omega}) = \begin{bmatrix}
0 & -\omega_x & -\omega_y & -\omega_z \\\\
\omega_x & 0 & \omega_z & -\omega_y \\\\
\omega_y & -\omega_z & 0 & \omega_x \\\\
\omega_z & \omega_y & -\omega_x & 0
\end{bmatrix}
$$

Then normalize: $\mathbf{q}_{k+1} \leftarrow \frac{\mathbf{q}_{k+1}}{\|\mathbf{q}_{k+1}\|}$

### Velocity Update

Transform acceleration to world frame and remove gravity:

$$
\mathbf{a}_{world} = R(\mathbf{q}_k) \mathbf{a} - \mathbf{g}_{NED}
$$

Where $\mathbf{g}_{NED} = [0, 0, 9.81]^T$ m/s²

Then update velocity:

$$
\mathbf{v}_{k+1} = \mathbf{v}_k + \mathbf{a}_{world} \Delta t
$$

### Position Update

$$
\mathbf{p}_{k+1} = \mathbf{p}_k + \mathbf{v}_k \Delta t + \frac{1}{2} \mathbf{a}_{world} \Delta t^2
$$

### Bias Update (Random Walk Model)

Biases drift slowly, modeled as Brownian motion:

$$
\mathbf{b}_{a,k+1} = \mathbf{b}_{a,k} + \mathbf{w}_a
$$

$$
\mathbf{b}_{g,k+1} = \mathbf{b}_{g,k} + \mathbf{w}_g
$$

Where $\mathbf{w}_a \sim \mathcal{N}(0, \sigma_a^2)$ and $\mathbf{w}_g \sim \mathcal{N}(0, \sigma_g^2)$ are process noise.

### Covariance Propagation

$$
\mathbf{P}_{k+1} = \mathbf{F}_k \mathbf{P}_k \mathbf{F}_k^T + \mathbf{Q}_k
$$

Where:
- $\mathbf{F}_k$ = Jacobian of state transition (16×16 matrix)
- $\mathbf{Q}_k$ = Process noise covariance (accounts for IMU noise and model uncertainty)

## 4.3 Update Step: TLIO Measurement (2Hz)

Every 0.5 seconds, TLIO provides a displacement estimate. The EKF uses this to correct accumulated drift.

### Measurement Model

TLIO predicts **displacement in body frame**:

$$
\mathbf{z}_{TLIO} = \boldsymbol{\Delta p}_{body} + \mathbf{v}_{meas}
$$

Where $\mathbf{v}_{meas} \sim \mathcal{N}(0, \mathbf{R}_{TLIO})$ is measurement noise.

**Measurement covariance** (from TLIO uncertainty):

$$
\mathbf{R}_{TLIO} = \begin{bmatrix}
\exp(\log \sigma_x^2) & 0 & 0 \\\\
0 & \exp(\log \sigma_y^2) & 0 \\\\
0 & 0 & \exp(\log \sigma_z^2)
\end{bmatrix}
$$

This is the **adaptive uncertainty** learned by the neural network!

### Innovation (Prediction Error)

Compute what the EKF *predicts* the displacement should be:

$$
\mathbf{z}_{pred} = R(\mathbf{q}_k)^T (\mathbf{p}_k - \mathbf{p}_{0.5s\\_ago})
$$

Then compute innovation:

$$
\mathbf{y} = \mathbf{z}_{TLIO} - \mathbf{z}_{pred}
$$

### Kalman Gain

$$
\mathbf{K} = \mathbf{P}_k \mathbf{H}^T (\mathbf{H} \mathbf{P}_k \mathbf{H}^T + \mathbf{R}_{TLIO})^{-1}
$$

Where:
- $\mathbf{H}$ = Measurement Jacobian (3×16 matrix, relates state to measurement)
- $\mathbf{K}$ = Kalman gain (16×3 matrix)

**Interpretation of Kalman Gain:**

When TLIO is **confident** (small $\mathbf{R}_{TLIO}$):
- $\mathbf{K}$ is large → trust TLIO measurement more
- State correction is aggressive

When TLIO is **uncertain** (large $\mathbf{R}_{TLIO}$):
- $\mathbf{K}$ is small → trust IMU dead-reckoning more
- State correction is gentle

### State Update

$$
\mathbf{x}_{k+1} = \mathbf{x}_k + \mathbf{K} \mathbf{y}
$$

This corrects position, velocity, orientation, and bias estimates simultaneously!

### Covariance Update

$$
\mathbf{P}_{k+1} = (\mathbf{I} - \mathbf{K} \mathbf{H}) \mathbf{P}_k
$$

Uncertainty decreases after measurement update.

## 4.4 Update Step: Barometer (10Hz)

The barometer provides absolute altitude measurements, preventing Z-axis drift.

### Measurement Model

$$
z_{baro} = -p_z + v_{baro}
$$

Where:
- $-p_z$ = altitude (negative of down position)
- $v_{baro} \sim \mathcal{N}(0, \sigma_{baro}^2)$ = barometer noise (~0.5m std)

**Measurement covariance:**

$$
R_{baro} = \sigma_{baro}^2 \approx 0.25 \text{ m}^2
$$

### Innovation

$$
y_{baro} = z_{baro} - (-p_z)
$$

### Kalman Gain (Simplified for 1D)

Only affects the Z-position component:

$$
K_z = \frac{P_{zz}}{P_{zz} + R_{baro}}
$$

Where $P_{zz}$ is the Z-position variance from the covariance matrix.

### State Update

$$
p_z \leftarrow p_z + K_z \cdot y_{baro}
$$

**Why barometer for Z but not TLIO?**

- Barometer: Absolute altitude reference, drift-free
- TLIO Z-prediction: Relative displacement, accumulates drift
- Configuration: `EK3_SRC1_POSZ = 3` (Barometer) instead of 6 (GHOST)

# 5. Data Pipeline: From Raw Sequences to Training Samples

This section explains how we process the TLIO dataset to create training examples.

## 5.1 Sliding Window Creation

### Window Parameters

- **Window size**: 250 samples (1.25 seconds @ 200Hz)
- **Stride**: 100 samples (0.5 seconds)
- **Overlap**: 60% (150 samples)

### Mathematical Formulation

Given a sequence with $N$ IMU samples:

$$
\text{num\_windows} = \left\lfloor \frac{N - \text{window\_size}}{\text{stride}} \right\rfloor + 1
$$

**Example:**
- Sequence length: $N = 1000$ samples
- Window size: 250
- Stride: 100

$$
\text{num\_windows} = \left\lfloor \frac{1000 - 250}{100} \right\rfloor + 1 = \left\lfloor 7.5 \right\rfloor + 1 = 8
$$

### Window Extraction

For window $i$:

$$
\text{start\_idx} = i \times \text{stride}
$$

$$
\text{end\_idx} = \text{start\_idx} + \text{window\_size}
$$

$$
\mathbf{X}_i = \text{IMU}[\text{start\_idx} : \text{end\_idx}, :]
$$

$$
\mathbf{y}_i = \text{displacement from start to end of window}
$$

In [ ]:
# Example: Load and visualize real TLIO data
import numpy as np
import matplotlib.pyplot as plt

# This would load actual TLIO data using load_data_tf.py
# from load_data_tf import load_tlio_sequence

# For demonstration, we'll show the structure:
print("TLIO Data Structure:")
print("-" * 50)
print("IMU shape: (N, 6) where N = number of samples")
print("  - Columns: [acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z]")
print("  - Units: m/s² for accel, rad/s for gyro")
print("  - Frequency: 200 Hz")
print()
print("Position shape: (N, 3)")
print("  - Columns: [x, y, z] in world frame (meters)")
print("  - Ground truth from motion capture or SLAM")
print()
print("Quaternion shape: (N, 4)")
print("  - Columns: [qw, qx, qy, qz]")
print("  - Orientation at each timestep")
print()
print("After sliding window:")
print("  X_train shape: (num_windows, 250, 6)")
print("  y_train shape: (num_windows, 3)  # body-frame displacement")

# 6. Training Process

## 6.1 Training Configuration

### Optimizer

Adam optimizer with learning rate schedule:

$$
\text{lr}(epoch) = \begin{cases}
0.001 & \text{epochs } 0-20 \\\\
0.0005 & \text{epochs } 21-40 \\\\
0.0001 & \text{epochs } 41+
\end{cases}
$$

### Batch Size

- **Training**: 64 samples per batch
- **Validation**: 128 samples per batch

### Loss Function

NLL loss (implemented per-axis):

```python
def nll_loss(y_true, y_pred):
    displacement_pred = y_pred[:, :3]  # [dx, dy, dz]
    log_var = y_pred[:, 3:]            # [log_σx², log_σy², log_σz²]
    
    variance = tf.exp(log_var)
    loss = 0.5 * (tf.square(displacement_pred - y_true) / variance + log_var)
    return tf.reduce_mean(loss)
```

### Metrics Monitored

1. **Displacement MAE**: Mean Absolute Error in meters
   $$
   \text{MAE} = \frac{1}{N} \sum_{i=1}^{N} |\boldsymbol{\Delta p}_{pred,i} - \boldsymbol{\Delta p}_{true,i}|
   $$

2. **NLL Loss**: For uncertainty calibration

3. **Per-Axis MAE**: Separate tracking for X, Y, Z axes

### Early Stopping

- Monitor: `val_displacement_mae`
- Patience: 30 epochs
- Restore best weights

# 7. Performance Analysis

## 7.1 Architecture Comparison

| Architecture | MAE (m) | Parameters | STM32H7 Latency | Deployment |
|--------------|---------|------------|-----------------|------------|
| **v1 (ResNet-1D)** | 0.20-0.25 | ~50K | 8-12ms | ✅ TFLM |
| **v2 (FFT + Multi-Scale)** | 0.13-0.15 | ~70K | N/A | ❌ FFT not supported |
| **v2.5 (Dilated + Multi-Scale)** | 0.14-0.16 | ~70K | 10-20ms | ✅ TFLM |

**Key Insights:**

- **v1**: Baseline, simple ResNet-1D architecture
- **v2**: Best accuracy (+40% vs v1), but FFT prevents TFLM deployment
- **v2.5**: Near-v2 accuracy (90-95%), fully TFLM-compatible

## 7.2 Drift Characteristics

### Mathematical Model

Drift accumulates as **random walk**:

$$
\text{drift}(t) \approx \text{MAE} \times \sqrt{N_{predictions}}
$$

Where:
- $N_{predictions} = \frac{t}{0.5s}$ (predictions at 2Hz)
- $t$ = flight time in seconds

### Example: 10-Minute Flight

$$
N_{predictions} = \frac{600s}{0.5s} = 1200
$$

With v2.5 MAE = 0.14m:

$$
\text{drift} \approx 0.14 \times \sqrt{1200} \approx 0.14 \times 34.6 \approx 4.8 \text{ m}
$$

**In practice**: 10-15m drift due to correlated errors (turns, vibrations).

### Drift is Time-Dependent, NOT Distance-Dependent

**Common misconception**: \"Drift depends on how far you fly\"

**Reality**: Drift depends on **how long** you fly:
- Hovering in place for 10 min: ~10m drift
- Flying 1 km in 10 min: ~10m drift (same!)

Why? Each prediction has independent error that accumulates over time.

## 7.3 Inference Timing

| Platform | v1 | v2.5 | Notes |
|----------|-----|------|-------|
| CPU (i7-10700) | 2-3ms | 3-5ms | Python + TensorFlow |
| STM32H7 (480MHz) | 8-12ms | 10-20ms | TFLM C++ (estimated) |

**Real-time requirement**: <500ms for 2Hz updates → ✅ Both architectures have 20×+ margin

# 8. Hardware Deployment

## 8.1 ArduPilot EKF3 Integration

TLIO integrates with ArduPilot through the **VISION_POSITION_ESTIMATE** MAVLink message, exactly like external motion capture systems (Vicon, OptiTrack).

### MAVLink Message

```python
mav.mav.vision_position_estimate_send(
    usec=timestamp,                 # Microseconds since boot
    x=float(pos_ekf[0]),           # North (m)
    y=float(pos_ekf[1]),           # East (m)
    z=float(pos_ekf[2]),           # Down (m)
    roll=float(euler[0]),          # Roll (rad)
    pitch=float(euler[1]),         # Pitch (rad)
    yaw=float(euler[2]),           # Yaw (rad)
    covariance=[0.01]*21           # Covariance matrix (optional)
)
```

### ArduPilot Configuration

See Section 1 for complete `EK3_SRC1_*` parameter settings.

**Critical parameters:**

```
EK3_SRC1_POSXY = 6    # Use ExternalNav (GHOST) for horizontal position
EK3_SRC1_POSZ  = 3    # Use Barometer for altitude
GPS_TYPE       = 0    # Disable GPS
VISO_TYPE      = 1    # Enable MAVLink vision estimates
```

## 8.2 Real-World Impact

### Before TLIO/GHOST

- **GPS-denied navigation**: Impossible without external infrastructure
- **Indoor flight**: Requires Vicon ($100K+) or OptiTrack ($50K+)
- **GPS jamming**: Drone enters failsafe, lands or crashes

### After TLIO/GHOST

- **GPS-denied navigation**: Works out-of-the-box with onboard IMU
- **Indoor flight**: No external sensors needed
- **GPS jamming**: Seamless fallback to TLIO, mission continues

### Drift Tolerance

For most applications, 10-15m drift over 10 minutes is **acceptable**:

- **Search & rescue**: Find general area, then manual control
- **Indoor delivery**: Navigate to room, then precision landing with downward camera
- **Racing drones**: Lap times ~30-60 seconds → drift <1m

## 8.3 Hardware Requirements

### Minimum

- **Flight Controller**: Any STM32H7-based (2MB flash, 1MB RAM)
  - Examples: Pixhawk 6C, Holybro H7, CUAV X7
- **Companion Computer**: For running TFLM inference
  - Raspberry Pi 4 (overkill)
  - ESP32-S3 (sufficient for v1)
  - Custom STM32H7 board (optimal)

### Recommended

- **Flight Controller**: Integrated with companion computer
- **IMU**: Invensense ICM-42688 or ICM-20948 (200Hz capable)
- **Barometer**: MS5611 or BMP388 (±0.5m accuracy)

### Power Consumption

- TFLM inference @ 2Hz: ~50-100mW additional draw
- Negligible impact on flight time (<1%)

---

## Summary: The Complete Mathematical Pipeline

```
Raw IMU (200Hz)
    ↓
[Normalization: acc/9.81, gyro/π]
    ↓
[Sliding Windows: 250 samples = 1.25s]
    ↓
[Neural Network v2.5]
 • Input: (250, 6)
 • Multi-Scale Convolutions (kernel 3/7/15)
 • Dilated Convolutions (dilation 1/2/4/8)
 • SE Attention
 • Output: (6) = [dx, dy, dz, log_σx², log_σy², log_σz²]
    ↓
[NLL Loss Training]
 • L = 0.5 * (error²/σ² + log(σ²))
 • Learns both accuracy AND uncertainty
    ↓
[Coordinate Transform]
 • Body frame → World frame
 • Using quaternion rotation matrix
    ↓
[EKF Sensor Fusion]
 • PREDICT: 200Hz with IMU (fast, drifts)
 • UPDATE: 2Hz with TLIO (corrects drift)
 • UPDATE: 10Hz with Baro (altitude reference)
 • Kalman gain auto-balances based on TLIO uncertainty
    ↓
[Position, Velocity, Attitude]
 • Sent to PID controller
 • Enables GPS-free autonomous flight
```

**Key Innovations:**

1. **NLL Loss**: Uncertainty quantification enables adaptive sensor fusion
2. **Dilated Convolutions**: TFLM-compatible replacement for FFT (90-95% accuracy)
3. **Body Frame Prediction**: Natural for CNN to learn from IMU patterns
4. **GHOST Integration**: Seamless drop-in replacement for GPS in ArduPilot

**Result**: GPS-free navigation with 10-15m drift over 10 minutes, deployable on $50 hardware. 🚀

# CubeOrange+ Deployment Steps
## What you already have (done)
```
tlio_model_data.h — v2.5 float32 model embedded
AP_GHOST_config.h — arena 200KB, W=200
AP_GHOST.cpp — all 3 bugs fixed
```
### Step 1 — Set up WSL2 + ArduPilot (one-time)
```
# In PowerShell as Admin:
wsl --install
# Restart PC, then open Ubuntu terminal:

cd ~
git clone --recurse-submodules https://github.com/ArduPilot/ardupilot.git
cd ardupilot
Tools/environment_install/install-prereqs-ubuntu.sh -y
. ~/.profile
```
### Step 2 — Bundle TensorFlow Lite Micro into AP_GHOST
TFLM is not included in ArduPilot by default — you must copy it in.
```
cd /mnt/x/fpv_AI_agent/backup/manageable_version_last/rl_autopilot_option/EKF2/ardupilot_GHOST/AP_GHOST

git clone --depth=1 https://github.com/tensorflow/tflite-micro.git tflm_temp

mkdir -p tensorflow/lite
cp -r tflm_temp/tensorflow/lite/micro    tensorflow/lite/
cp -r tflm_temp/tensorflow/lite/core     tensorflow/lite/
cp -r tflm_temp/tensorflow/lite/schema   tensorflow/lite/
cp -r tflm_temp/tensorflow/lite/kernels  tensorflow/lite/

rm -rf tflm_temp
```
### Step 3 — Copy AP_GHOST into ArduPilot source tree
```
cp -r /mnt/x/fpv_AI_agent/backup/manageable_version_last/rl_autopilot_option/EKF2/ardupilot_GHOST/AP_GHOST \
      ~/ardupilot/libraries/AP_GHOST

# Verify:
ls ~/ardupilot/libraries/AP_GHOST/
# Must show: AP_GHOST.h  AP_GHOST.cpp  AP_GHOST_config.h  tlio_model_data.h  tensorflow/
```
### Step 4 — Integrate into ArduCopter (3 file edits)
```
A. ~/ardupilot/ArduCopter/Copter.h — add near the other library includes:


#if AP_GHOST_ENABLED
#include <AP_GHOST/AP_GHOST.h>
#endif
Add in the class Copter private section:


#if AP_GHOST_ENABLED
    AP_GHOST ghost;
#endif
B. ~/ardupilot/ArduCopter/system.cpp — inside Copter::init_ardupilot():


#if AP_GHOST_ENABLED
    ghost.init();
#endif
C. ~/ardupilot/ArduCopter/Copter.cpp — inside scheduler_tasks[]:


#if AP_GHOST_ENABLED
    SCHED_TASK_CLASS(AP_GHOST, &copter.ghost, update, 200, 500, 250),
#endif
```
### Step 5 — Build for CubeOrange+
```
cd ~/ardupilot
./waf configure --board CubeOrange
./waf copter
Build takes ~10-20 min. Output: build/CubeOrange/bin/arducopter.apj

If it fails with AllocateTensors → increase arena in AP_GHOST_config.h to (256 * 1024), rebuild.
```
### Step 6 — Flash via Mission Planner (Windows)
Connect CubeOrange+ to PC via USB
Open Mission Planner → Setup → Install Firmware
Click "Load custom firmware"
Browse to: 
``` \\wsl$\Ubuntu\home\<your_user>\ardupilot\build\CubeOrange\bin\arducopter.apj 
```
Wait for "Upload Done"
Board reboots automatically
```
Or from WSL directly (board connected):


./waf copter --upload
```
### Step 7 — Set EKF3 parameters (Mission Planner or MAVProxy)
```
param set AHRS_EKF_TYPE   3
param set VISO_TYPE        1
param set EK3_SRC1_POSXY  6
param set EK3_SRC1_POSZ   6
param set EK3_SRC1_VELXY  0
param set EK3_SRC1_VELZ   0
param set EK3_SRC1_YAW    1
param set GPS_TYPE         0
param set EK3_GPS_TYPE     3
param set EK3_POSNE_M_NSE  0.09
param save
reboot

EK3_POSNE_M_NSE 0.09 = your model's mean 3D error (~0.09m). Increase to 0.15 if the drone oscillates.
```
### Step 8 — First flight checklist
After boot, check GCS messages — you should see:

```
GHOST: Initialized — GPS-free TLIO active
GHOST TFLM: Input [1,200,6] Output [1,6]
GHOST TFLM: Arena xxxxx / 204800 bytes
```
```
Step	Action
1	Set EKF origin (required — no GPS): wp setorigin <lat> <lon> <alt>
2	Arm in Stabilize mode
3	Takeoff manually — confirm stable hover
4	Switch to AltHold → confirm altitude hold
5	Switch to Loiter → GHOST holds XY position
6	Monitor: GHOST #20: pos=[...]m inf=XXms — inference should be ~80-150ms
```
### Flash budget check (tight but fits)
```
Component	Size
ArduCopter firmware	~1250 KB
GHOST v2.5 float32 model	668 KB
Total	~1918 KB / 2048 KB
Margin	~130 KB
```
```
If flash is full → go back and use the int8 path (needs calibration data, ~170KB model instead of 668KB, gives ~628KB margin).
```